![alt text](../../pythonexposed-high-resolution-logo-black.jpg "Optionele titel")

In [1]:
import pandas as pd
import numpy as np

# Generate dataset with 730 dates and random values between 60 and 80
date_range = pd.date_range(start="2023-01-01", end="2024-12-31", freq='D')
values = np.random.randint(60, 81, size=len(date_range))

# Create DataFrame
df = pd.DataFrame({
    "Date": date_range,
    "Value": values
})

# Save to CSV
file_path = "data/dataset_730_days.csv"
df.to_csv(file_path, index=False)

In [2]:
# Code to read the CSV and create a MultiIndex DataFrame
def create_multiindex_dataframe(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])

    # Extract hierarchy levels
    df['Year'] = df['Date'].dt.year
    df['Season'] = df['Date'].dt.month % 12 // 3 + 1  # Winter: 1, Spring: 2, Summer: 3, Fall: 4
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day

    # Create MultiIndex
    df.set_index(['Year', 'Season', 'Month', 'Day'], inplace=True)
    return df

# Generate the trend column
def add_trend_column(df):
    df['Trend'] = np.where(df['Value'].diff().fillna(0) > 0, "stijgend", "dalend")
    return df

# Example usage
multiindex_df = create_multiindex_dataframe(file_path)
multiindex_df = add_trend_column(multiindex_df)
multiindex_df.head()

Date  Value     Trend
Year Season Month Day                            
2023 1      1     1   2023-01-01     74    dalend
                  2   2023-01-02     68    dalend
                  3   2023-01-03     74  stijgend
                  4   2023-01-04     61    dalend
                  5   2023-01-05     80  stijgend

In [3]:
# Filter alle waarden in de zomer waar de trend 'stijgend' is
filtered_df = multiindex_df.loc[
    (multiindex_df.index.get_level_values('Season') == 3) &  # Summer
    (multiindex_df['Trend'] == 'stijgend')
]

# Resultaat bekijken
print(filtered_df)


                            Date  Value     Trend
Year Season Month Day                            
2023 3      6     1   2023-06-01     67  stijgend
                  3   2023-06-03     64  stijgend
                  4   2023-06-04     70  stijgend
                  5   2023-06-05     80  stijgend
                  7   2023-06-07     75  stijgend
...                          ...    ...       ...
2024 3      8     22  2024-08-22     61  stijgend
                  23  2024-08-23     63  stijgend
                  25  2024-08-25     75  stijgend
                  27  2024-08-27     72  stijgend
                  29  2024-08-29     65  stijgend

[87 rows x 3 columns]


De berekening van het seizoen werkt met deze regel:

```python
df['Season'] = df['Date'].dt.month % 12 // 3 + 1
```

Dit is een compacte manier om een maand naar een seizoen toe te wijzen. Hier is de stap-voor-stap uitleg van hoe het werkt:

---

### 1. **Maanwaarden en seizoenen**
De maanden worden verdeeld over de vier seizoenen:
- **Winter (1)**: December (12), Januari (1), Februari (2)
- **Lente (2)**: Maart (3), April (4), Mei (5)
- **Zomer (3)**: Juni (6), Juli (7), Augustus (8)
- **Herfst (4)**: September (9), Oktober (10), November (11)

Het doel is om de maandnummers naar deze seizoenen te converteren.

---

### 2. **De formule in delen**

#### (a) `df['Date'].dt.month`
- Dit haalt de maand op uit de datum. Bijvoorbeeld:
  - Januari = 1
  - Maart = 3
  - September = 9

#### (b) `% 12`
- De modulo-operator `% 12` zorgt ervoor dat de waarde binnen het bereik van 0 tot 11 blijft. Hierdoor wordt December (12) naar 0 omgezet.
  - Voorbeeld:
    - 12 % 12 = 0 (December wordt 0)
    - 1 % 12 = 1 (Januari blijft 1)
    - 9 % 12 = 9 (September blijft 9)

#### (c) `// 3`
- De integer-division `// 3` verdeelt de maanden in groepen van drie:
  - Maanden 0-2 → 0 (Winter)
  - Maanden 3-5 → 1 (Lente)
  - Maanden 6-8 → 2 (Zomer)
  - Maanden 9-11 → 3 (Herfst)

#### (d) `+ 1`
- Door 1 op te tellen, worden de groepen aangepast van 0-3 naar 1-4:
  - Winter: 1
  - Lente: 2
  - Zomer: 3
  - Herfst: 4

In [7]:
# Reload the file with the newly uploaded dataset and proceed with processing
file_path = "data/dataset_730_days.csv"

# Modify the functions to add day of the week and weekend/weekday columns
def create_multiindex_dataframe_with_weekday(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])

    # Extract hierarchy levels
    df['Year'] = df['Date'].dt.year
    df['Season'] = df['Date'].dt.month % 12 // 3 + 1  # Winter: 1, Spring: 2, Summer: 3, Fall: 4
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['Day_of_Week'] = df['Date'].dt.day_name()  # Add the name of the day
    df['Weekday_Weekend'] = np.where(df['Date'].dt.dayofweek < 5, "Weekday", "Weekend")  # Weekday/Weekend

    # Create MultiIndex
    df.set_index(['Year', 'Season', 'Month', 'Day'], inplace=True)
    return df

# Reuse the trend function
def add_trend_column(df):
    df['Trend'] = np.where(df['Value'].diff().fillna(0) > 0, "stijgend", "dalend")
    return df

# Example usage
multiindex_df = create_multiindex_dataframe_with_weekday(file_path)
multiindex_df = add_trend_column(multiindex_df)

# Filter: All Mondays in summer of the first year where the trend is "stijgend"
first_year = multiindex_df.index.get_level_values('Year').min()
filtered_df = multiindex_df.loc[
    (multiindex_df.index.get_level_values('Year') == first_year) &
    (multiindex_df.index.get_level_values('Season') == 3) &  # Summer
    (multiindex_df['Day_of_Week'] == 'Monday') &
    (multiindex_df['Trend'] == 'stijgend')
]

# Create a new dataframe with weekly averages
weekly_avg_df = multiindex_df.reset_index().set_index('Date').resample('W').agg({'Value': 'mean'}).rename(columns={'Value': 'Weekly_Average'})

weekly_avg_df

,Weekly_Average
Date,
2023-01-01,65.000000
2023-01-08,68.285714
2023-01-15,70.714286
2023-01-22,69.285714
2023-01-29,71.285714
...,...
2024-12-08,71.285714
2024-12-15,67.857143
2024-12-22,69.428571
